# INF01090 - Ciência de Dados

# Lab Task 01 — Visualization Techniques with Altair using your own dataset (4 people)

Choose an interesting dataset to explore and produce visualizations in a similar fashion described in the preparation notebook. Visualization is not only about coding, it is about making **good analytical and design choices**.

Deliver the results as a notebook, which can run in other machines. Include the dataset with the submission if it can not be downloaded from a repository. 

## Goals

- choose an interesting dataset
- adapt the ideas described in the preparatory notebook to your dataset
- explain the results and interesting visualizations obtained from your dataset 
- feel free to add more advanced analysis and algorithms, that help better understand the dataset
- use vibe coding extensively (when it makes sense) to improve the final results
- **deliver a notebook with the solution using Moodle** (include your dataset if not able to download directly from the notebook) 

## Awards
- the top 3 ranked solutions will be asked to demonstrate the results in a subsequent lab class

## 1. Choose your own dataset 

### Good dataset criteria

Choose a dataset that is:
- tabular
- not too large
- understandable
- rich enough to support multiple questions

It should ideally contain:
- at least one numerical variable
- at least one categorical variable
- optionally a time variable

### Examples

- sports statistics
- movies or streaming data
- flights
- public health data
- environmental data
- education data

### Checklist for your own dataset

Before creating charts, answer these questions:

1. What is one interesting question I want to answer?
2. Which columns are numerical?
3. Which columns are categorical?
4. Is there a time column?
5. Are there missing values?
6. Which chart type best matches the question?
7. What should the viewer learn from the chart?

## 2. Load and inspect the data



In [1]:
%pip install pandas altair vega_datasets scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\crisw\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
# Importando as bibliotecas necessárias (conforme o Lab Preparation)
import pandas as pd
import altair as alt

# Desabilitando o limite de linhas para evitar erros com datasets maiores
alt.data_transformers.disable_max_rows()

# 1. Carregando os dados
# Substitua 'foodpanda.csv' pelo nome exato do arquivo que vocês baixaram
df = pd.read_csv('student_productivity_distraction_dataset_20000.csv')

# 2. Inspecionando o dataset (Respondendo ao checklist da Etapa 1)
print(f"O dataset possui {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

# 3. Verificando dados duplicados baseado no student_id 
num_duplicates = df.duplicated(subset=['student_id']).sum()
print(f"Número de linhas duplicadas: {num_duplicates}\n")

print("--- Informações das Colunas e Tipos de Dados ---")
df.info()

print("\n--- Valores Nulos por Coluna ---")
print(df.isna().sum())

# Mostrando as 5 primeiras linhas para entender a "cara" dos dados
df.head()

O dataset possui 20000 linhas e 18 colunas.

Número de linhas duplicadas: 0

--- Informações das Colunas e Tipos de Dados ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             20000 non-null  int64  
 1   age                    20000 non-null  int64  
 2   gender                 20000 non-null  object 
 3   study_hours_per_day    20000 non-null  float64
 4   sleep_hours            20000 non-null  float64
 5   phone_usage_hours      20000 non-null  float64
 6   social_media_hours     20000 non-null  float64
 7   youtube_hours          20000 non-null  float64
 8   gaming_hours           20000 non-null  float64
 9   breaks_per_day         20000 non-null  int64  
 10  coffee_intake_mg       20000 non-null  int64  
 11  exercise_minutes       20000 non-null  int64  
 12  assignments_completed  20000 non

,student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score
0,1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78
1,2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.90,48.99
2,3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.60
3,4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87
4,5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.90


## 2.1 Processamento e Geração das Amostras

In [3]:
import pandas as pd
import altair as alt
import numpy as np

# Desabilitar max rows
alt.data_transformers.disable_max_rows()

# =====================================================================
# 1. GERANDO AS 10 AMOSTRAS (Literatura Acadêmica) - ~1000 linhas cada
# =====================================================================

# 01. População (Baseline)
df_pop = df.copy()
df_pop['Método'] = '01. População (20000)'

# 02. Aleatória Simples (Simple Random)
df_simples = df.sample(n=1000, random_state=42).copy()
df_simples['Método'] = '02. Aleatória Simples'
# 03. Estratificada (Stratified) - Estratificando por gênero para garantir representação
df_estratificada = df.groupby('gender', group_keys=False).apply(lambda x: x.sample(n=333, random_state=42, replace=True)).copy()
df_estratificada['Método'] = '03. Estratificada (50/50 Gênero)'
# 04. Sistemática (Systematic) - Pula de 10 em 10 registros
df_sistematica = df.iloc[::20].copy()
df_sistematica['Método'] = '04. Sistemática (Passo 10)'

# 07. PPS - Probabilidade Proporcional ao Tamanho (Proportional to Size)
# Pedidos mais caros têm maior chance matemática de serem sorteados.
df_pps = df.sample(n=1000, weights='productivity_score', random_state=42).copy()
df_pps['Método'] = '07. PPS (Peso: Productivity)'

# 08. Conveniência (Convenience) - Pega as primeiras 1000 linhas (Não-probabilístico)
df_conveniencia = df.head(1000).copy()
df_conveniencia['Método'] = '08. Conveniência (Head)'

# Juntando tudo no DataFrame de comparação
df_comparacao = pd.concat([
    df_pop, df_simples, df_estratificada, df_sistematica, df_pps, df_conveniencia
])

# =====================================================================
# 1.5 OUTPUT EM TEXTO: RESUMO DAS AMOSTRAS E SANITY CHECK
# =====================================================================

# Agrupando os dados para mostrar o tamanho de cada amostra e suas médias
resumo_texto = df_comparacao.groupby('Método').agg(
    Qtd_Linhas=('student_id', 'count'),
    Estudio_Médio=('study_hours_per_day', 'mean'),
    Presenca_Média=('attendance_percentage', 'mean')
).round(2) # Arredonda para 2 casas decimais para ficar bonito

# O display() no Jupyter formata a tabela como HTML bonitinho (melhor que print)
display(resumo_texto)

C:\Users\crisw\AppData\Local\Temp\ipykernel_26312\1316365297.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_estratificada = df.groupby('gender', group_keys=False).apply(lambda x: x.sample(n=333, random_state=42, replace=True)).copy()


,Qtd_Linhas,Estudio_Médio,Presenca_Média
Método,,,
01. População (20000),20000,5.25,69.95
02. Aleatória Simples,1000,5.12,70.09
03. Estratificada (50/50 Gênero),999,5.47,69.61
04. Sistemática (Passo 10),1000,5.45,70.38
07. PPS (Peso: Productivity),1000,6.03,71.13
08. Conveniência (Head),1000,5.22,68.57


## 2.2 Escolha Acadêmica do Melhor Método de Amostragem

In [4]:
# =====================================================================
# DEFINIÇÃO DA AMOSTRA FINAL E LIMPEZA
# =====================================================================

# =====================================================================
# 3.3 AVALIAÇÃO MATEMÁTICA E SELEÇÃO ALGORÍTMICA
# =====================================================================
print("\n⚙️ Rodando Algoritmo de Validação Estatística (TVD e MAPE)...")

# 1. Calculando os baselines da População (Verdade Absoluta)
pop_productivity_mean = df_pop['productivity_score'].mean()
pop_attendance_mean = df_pop['attendance_percentage'].mean()
pop_cat_prop = df_pop['age'].value_counts(normalize=True)
pop_gender_prop = df_pop['gender'].value_counts(normalize=True)

resultados = []

# 2. Loop para avaliar cada método de amostragem
for metodo in df_comparacao['Método'].unique():
    if 'População' in metodo: 
        continue # Pula a base original
        
    df_m = df_comparacao[df_comparacao['Método'] == metodo]
    
    # Cálculo do MAPE para variáveis numéricas
    err_productivity = abs(df_m['productivity_score'].mean() - pop_productivity_mean) / pop_productivity_mean
    err_attendance = abs(df_m['attendance_percentage'].mean() - pop_attendance_mean) / pop_attendance_mean
    
    # Cálculo do TVD para variáveis categóricas (garantindo alinhamento de índices)
    m_cat_prop = df_m['age'].value_counts(normalize=True).reindex(pop_cat_prop.index, fill_value=0)
    tvd_cat = 0.5 * np.sum(np.abs(pop_cat_prop - m_cat_prop))
    
    m_gender_prop = df_m['gender'].value_counts(normalize=True).reindex(pop_gender_prop.index, fill_value=0)
    tvd_gender = 0.5 * np.sum(np.abs(pop_gender_prop - m_gender_prop))
    
    # Score Final: Soma das divergências (Quanto menor, melhor)
    divergencia_total = err_productivity + err_attendance + tvd_cat + tvd_gender
    
    resultados.append({
        'Método': metodo,
        'Erro Produtividade (MAPE)': err_productivity,
        'Erro Presença (MAPE)': err_attendance,
        'Erro Categórico (TVD)': tvd_cat,
        'Erro Gênero (TVD)': tvd_gender,
        'Divergência Total': divergencia_total
    })

# 3. Criando o DataFrame de Ranking e ordenando do melhor pro pior
df_ranking = pd.DataFrame(resultados).sort_values('Divergência Total').reset_index(drop=True)

print("🏆 RANKING DOS MÉTODOS DE AMOSTRAGEM (Menor erro vence):")
# Estilizando com background gradient (heatmap) para destacar os piores e melhores erros
display(df_ranking.style.background_gradient(subset=['Divergência Total'], cmap='RdYlGn_r'))

# =====================================================================
# DEFINIÇÃO DINÂMICA DA AMOSTRA FINAL E LIMPEZA
# =====================================================================

# O algoritmo extrai automaticamente o nome do vencedor
metodo_vencedor = df_ranking.iloc[0]['Método']
print(f"\n✅ Seleção Algorítmica Concluída! O método vencedor é: {metodo_vencedor}")

# Salvamos dinamicamente o dataset vencedor como o nosso principal
df_amostra = df_comparacao[df_comparacao['Método'] == metodo_vencedor].drop(columns=['Método']).copy()

print(f"Dimensões do dataset oficial de trabalho: {df_amostra.shape[0]} linhas e {df_amostra.shape[1]} colunas.")

# Limpeza de memória
del  df_simples, df_estratificada, df_sistematica
del  df_pps, df_conveniencia,df_comparacao, resultados



⚙️ Rodando Algoritmo de Validação Estatística (TVD e MAPE)...
🏆 RANKING DOS MÉTODOS DE AMOSTRAGEM (Menor erro vence):


,Método,Erro Produtividade (MAPE),Erro Presença (MAPE),Erro Categórico (TVD),Erro Gênero (TVD),Divergência Total
0,02. Aleatória Simples,0.010272,0.002031,0.030200,0.002100,0.044603
1,04. Sistemática (Passo 10),0.018567,0.006120,0.033800,0.005900,0.064387
2,08. Conveniência (Head),0.011469,0.019734,0.057100,0.007000,0.095303
3,07. PPS (Peso: Productivity),0.119161,0.016949,0.047200,0.012000,0.195311
4,03. Estratificada (50/50 Gênero),0.009927,0.004854,0.051520,0.294433,0.360734



✅ Seleção Algorítmica Concluída! O método vencedor é: 02. Aleatória Simples
Dimensões do dataset oficial de trabalho: 1000 linhas e 18 colunas.


# 2.3 Diagnóstico de Produtividade baseado no Nível de Stress e Horas de sono


In [5]:
import altair as alt

# Define a seleção (brush)
brush = alt.selection_interval()

# Gráfico 1: Dispersão (Sono vs Produtividade, Colorido por Café)
scatter = alt.Chart(df_amostra).mark_point(filled=True, size=60).encode(
    x=alt.X('sleep_hours:Q', title='Horas de Sono', scale=alt.Scale(zero=False)),
    y=alt.Y('productivity_score:Q', title='Score de Produtividade', scale=alt.Scale(zero=False)),
    # CORREÇÃO: Mudança de 'stress_level' para 'coffee_intake_mg' com legenda explícita
    color=alt.Color('coffee_intake_mg:Q', 
                    scale=alt.Scale(scheme='magma'), 
                    title='Ingestão de Café (mg)'),
    # Nova Interatividade: Pontos não selecionados ficam quase transparentes, mantendo a cor
    opacity=alt.condition(brush, alt.value(0.9), alt.value(0.1)),
    tooltip=['age', 'study_hours_per_day', 'coffee_intake_mg', 'stress_level']
).add_params(
    brush
).properties(
    width=450,
    height=350,
    title='Relação Sono, Café e Produtividade (Selecione uma área)'
)

# Gráfico 2: Barras Médias de Estresse filtradas pela seleção
bars = alt.Chart(df_amostra).mark_bar().encode(
    x=alt.X('stress_level:O', title='Nível de Estresse'),
    y=alt.Y('count()', title='Número de Estudantes na Seleção'),
    # Cor estática para o estresse
    color=alt.Color('stress_level:O', 
                    scale=alt.Scale(scheme='redyellowgreen', reverse=True), 
                    legend=None)
).transform_filter(
    brush # Mantém o filtro
).properties(
    width=350,
    height=350,
    title='Distribuição de Estresse (Filtrado)'
)

# Renderiza lado a lado
dashboard_final = scatter | bars
dashboard_final.properties(title='Dashboard Interativo: Sono, Café, Produtividade e Estresse')

alt.HConcatChart(...)

# 2.4 Análise do comportamento e distrações 

In [6]:
import altair as alt
import pandas as pd

# =====================================================================
# PREPARAÇÃO DOS DADOS (Mantendo o MELT que você gostou)
# =====================================================================

# Lista das colunas que consideramos distrações
distraction_vars = ['phone_usage_hours', 'social_media_hours', 'youtube_hours', 'gaming_hours']

# Usamos o .melt() para "dobrar" as colunas de distração
df_distractions = df_amostra.melt(
    id_vars=['student_id', 'focus_score', 'productivity_score'], # Colunas fixas
    value_vars=distraction_vars,                                  # Colunas "dobradas"
    var_name='tipo_distracao',
    value_name='horas_gastas'
)

# =====================================================================
# DEFINIÇÃO DAS INTERAÇÕES (DROPDOWN E BRUSH)
# =====================================================================

# 1. Seletor DROPDOWN para o menu de Distrações (Melt)
distraction_options = df_distractions['tipo_distracao'].unique().tolist()
dropdown = alt.binding_select(options=distraction_options, name='Escolha a Distração: ')
distraction_select = alt.selection_point(fields=['tipo_distracao'], bind=dropdown, value=distraction_options[0])

# 2. Seletor BRUSH para o nível de Foco (Histograma)
# Queremos selecionar apenas no eixo X do histograma
brush = alt.selection_interval(encodings=['x'])

# =====================================================================
# CONSTRUÇÃO DO PAINEL DE DOIS GRÁFICOS
# =====================================================================

# --- GRÁFICO A: O SELETOR DE FOCO (HISTOGRAMA) ---
# Usamos o brush para selecionar a faixa de foco.
# As barras mudam de cor se estiverem dentro da seleção.
hist_focus = alt.Chart(df_amostra).mark_bar().encode(
    x=alt.X('focus_score:Q', bin=True, title='Distribuição do Score de Foco'),
    y=alt.Y('count()', title='Número de Estudantes'),
    # CORREÇÃO VISUAL: Se estiver no brush, fica azul escuro, senão cinza claro.
    color=alt.condition(brush, alt.value('#2c3e50'), alt.value('lightgray'))
).properties(
    width=600,
    height=120,
    title='PASSO 1: Selecione uma faixa de Foco no histograma arrastando o mouse'
).add_params(
    brush # Adiciona a interação de brush neste gráfico
)


# --- GRÁFICO B: DISTRAÇÕES VS PRODUTIVIDADE COM LINHA DE TENDÊNCIA ---

# Definimos o gráfico básico (pontos neutros)
points = alt.Chart(df_distractions).mark_point(opacity=0.3, size=40, filled=True).encode(
    x=alt.X('horas_gastas:Q', title='Horas Gastas por Dia'),
    y=alt.Y('productivity_score:Q', title='Score de Produtividade'),
    color=alt.value('#606060') # Cor neutra para os pontos
)

# Definimos a linha de regressão dinâmica (esta linha é recalculada com base no filtro)
# Usamos mark_line para desenhá-la e transform_regression para calculá-la.
trend_line = points.transform_regression(
    'horas_gastas', 'productivity_score'
).mark_line(color='#c0392b', size=4).encode(
    tooltip=[
        alt.Tooltip('mean(horas_gastas):Q', title='Média de Horas (Selecção)', format='.1f'),
        alt.Tooltip('mean(productivity_score):Q', title='Produtividade Média (Selecção)', format='.1f')
    ]
)

# Camada final de Gráfico B (Pontos + Linha de Tendência)
# Adicionamos os filtros interativos e os seletores
main_distraction_plot = alt.layer(points, trend_line).add_params(
    distraction_select # Adiciona o menu dropdown
).transform_filter(
    distraction_select # Filtra pelo dropdown (tipo_distracao)
).transform_filter(
    brush # Filtra pelo brush (focus_score) que vem do gráfico A
).properties(
    width=600,
    height=350,
    title='PASSO 2: O gráfico principal e a linha de tendência dinâmica mostram a correlação para o foco selecionado'
)

# Juntamos os dois gráficos verticalmente (Seletor no topo, Principal na base)
dashboard_distractions_final = hist_focus & main_distraction_plot

dashboard_distractions_final.properties(title='Dashboard Interativo: Foco, Distrações e Produtividade')

alt.VConcatChart(...)

# 2.5 Relação de nota final e Produtividade

In [7]:
import altair as alt

# 1. Gráfico de dispersão puro: Produtividade vs Nota Final
pontos_choque = alt.Chart(df_amostra).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('productivity_score:Q', title='Score de Produtividade', scale=alt.Scale(zero=False)),
    y=alt.Y('final_grade:Q', title='Nota Final', scale=alt.Scale(zero=False)),
    color=alt.value('#e74c3c'), # Um vermelho forte para chamar a atenção
    tooltip=['productivity_score', 'final_grade']
)

# 2. A prova do crime: Linha de Regressão Linear
# Ela vai sair praticamente reta/horizontal, provando que o X não afeta o Y
linha_tendencia = pontos_choque.transform_regression(
    'productivity_score', 'final_grade'
).mark_line(
    color='#2c3e50', 
    size=4, 
    strokeDash=[5, 5] # Linha tracejada para ficar com cara de análise estatística
)

# 3. Junta os dois
grafico_impacto = (pontos_choque + linha_tendencia).properties(
    width=600,
    height=400,
    title='O Mito da Produtividade: Ausência Total de Correlação com a Nota Final'
)

grafico_impacto

alt.LayerChart(...)

# 2.6 Impacto do Desempenho Acadêmico

In [8]:
import altair as alt

# =====================================================================
# DEFINIÇÃO DAS INTERAÇÕES (BRUSH)
# =====================================================================

# Cria uma seleção intervalar (brush) que funciona nos dois gráficos
# Isso permite que a seleção em um gráfico filtre o outro automaticamente.
brush = alt.selection_interval()

# =====================================================================
# CONSTRUÇÃO DO NOVO PAINEL (Perfil Acadêmico vs Produtividade)
# =====================================================================

# --- GRÁFICO A: O NOVO MAPA DE PERFIL (Foco vs Presença) ---
# Usamos o 'mark_rect' para criar um heatmap binned, que resolve o 
# problema de ruído dos pontos, mostrando 'zonas' de densidade.
heatmap_perfil = alt.Chart(df_amostra).mark_rect().encode(
    # EIXOS ACADÊMICOS RELEVANTES: Foco e Presença
    x=alt.X('focus_score:Q', bin=alt.Bin(maxbins=10), title='Score de Foco (Faixas)'),
    y=alt.Y('attendance_percentage:Q', bin=alt.Bin(maxbins=10), title='Presença % (Faixas)'),
    # A COR É O NOSSO TARGET: Produtividade
    color=alt.Color('mean(productivity_score):Q', 
                    scale=alt.Scale(scheme='turbo'), # Paleta viva para destacar diferenças
                    title='Média de Produtividade'),
    # Adicionamos Tooltips detalhados
    tooltip=[
        alt.Tooltip('mean(focus_score):Q', title='Foco Médio (Região)', format='.1f'),
        alt.Tooltip('mean(attendance_percentage):Q', title='Presença Média (Região)', format='.1f'),
        alt.Tooltip('mean(productivity_score):Q', title='Produtividade Média', format='.1f'),
        alt.Tooltip('count()', title='Qtd. Alunos na Região')
    ]
).add_params(
    brush # Adiciona a interação de seleção
).properties(
    width=450,
    height=400,
    title='PASSO 1: Selecione uma zona Acadêmica (Foco x Presença)'
)


# --- GRÁFICO B: BOXPLOT DE ENTREGAS DINÂMICO ---
# Mostra a distribuição da produtividade baseada nos trabalhos entregues.
boxplot_entregas = alt.Chart(df_amostra).mark_boxplot(extent='min-max', size=30).encode(
    # Agrupamos as entregas para limpar o visual
    x=alt.X('assignments_completed:Q', bin=alt.Bin(maxbins=4), title='Trabalhos Entregues (Agrupado)'),
    # TARGET NO EIXO Y
    y=alt.Y('productivity_score:Q', title='Score de Produtividade', scale=alt.Scale(zero=False)),
    color=alt.Color('assignments_completed:Q', bin=alt.Bin(maxbins=4), legend=None, scale=alt.Scale(scheme='teals'))
).transform_filter(
    brush # MÁGICA: Este gráfico só mostra os dados selecionados no gráfico A!
).properties(
    width=350,
    height=400,
    title='PASSO 2: Produtividade por Entregas na Zona Selecionada'
)

# =====================================================================
# RENDERIZAÇÃO FINAL (Lado a Lado)
# =====================================================================
# Juntamos os gráficos com o operador '|'
painel_academico_fix = (heatmap_perfil | boxplot_entregas).resolve_scale(
    color='independent' # Permite que os gráficos tenham legendas de cor diferentes
).properties(
    title='Dashboard Interativo: Perfil Acadêmico Dinâmico e Produtividade'
)

painel_academico_fix

alt.HConcatChart(...)

# 3 Matriz de Correlação

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

# =====================================================================
# PREPARAÇÃO DOS DADOS: Cálculo da Matriz de Correlação
# =====================================================================

# 1. Selecionamos apenas as colunas numéricas (pois a correlação só funciona nelas)
df_numerico = df_amostra.select_dtypes(include=[np.number])

# 2. Calculamos a matriz de correlação puro (Pandas)
# .stack().reset_index() transforma a matriz 2D em um formato 'longo'
# (origem, destino, correlação) que o Altair consegue ler para fazer o heatmap.
df_correlacao = df_numerico.corr(method='pearson').stack().reset_index()
df_correlacao.columns = ['var_1', 'var_2', 'correlacao']

# =====================================================================
# CONSTRUÇÃO DO CORRELOGRAMA INTERATIVO (O Slide de Conclusões)
# =====================================================================

# O Gráfico Principal: Heatmap
# A cor representa a força e direção da correlação (azul = positivo, vermelho = negativo)
matriz_correlacao = alt.Chart(df_correlacao).mark_rect().encode(
    x=alt.X('var_1:N', title=None), # Nomes das variáveis no eixo X (Nominal)
    y=alt.Y('var_2:N', title=None), # Nomes das variáveis no eixo Y
    color=alt.Color('correlacao:Q', 
                    scale=alt.Scale(scheme='redblue', domain=[-1, 1], reverse=False),
                    title='Correlação de Pearson'),
    tooltip=[
        alt.Tooltip('var_1', title='Variável 1'),
        alt.Tooltip('var_2', title='Variável 2'),
        alt.Tooltip('correlacao', title='Índice de Correlação', format='.3f')
    ]
).properties(
    width=600,
    height=550,
    title='Correlograma de Pearson (Slide de Conclusões)'
)

# Renderiza o correlograma
matriz_correlacao

alt.Chart(...)

## Como o Tempo de Estudo e o Estresse afetam a Produtividade

In [ ]:
# 1. Definir a ferramenta de seleção (brush)
brush = alt.selection_interval(encodings=['x', 'y'])

# 2. Criar o Gráfico de Dispersão (Scatter Plot)
scatter = alt.Chart(df).mark_circle(opacity=0.4, size=30).encode(
    x=alt.X('study_hours_per_day:Q', title='Horas de Estudo por Dia'),
    y=alt.Y('final_grade:Q', title='Nota Final'),
    # A cor muda se o ponto estiver dentro da área selecionada pelo brush
    color=alt.condition(brush, alt.value('#2ca02c'), alt.value('lightgray')),
    tooltip=['study_hours_per_day', 'final_grade', 'stress_level', 'productivity_score']
).add_params(
    brush # Adicionando a interatividade ao gráfico
).properties(
    width=400,
    height=350,
    title='Relação: Horas de Estudo vs. Nota Final'
)

# 3. Criar o Gráfico de Barras (Bar Chart)
# Esse gráfico vai mostrar o nível de estresse no eixo X e a produtividade média no Y
bar = alt.Chart(df).mark_bar().encode(
    x=alt.X('stress_level:O', title='Nível de Estresse (0 a 10)'),
    y=alt.Y('mean(productivity_score):Q', title='Produtividade Média'),
    # Usando um esquema de cores laranja/vermelho para representar o estresse
    color=alt.Color('stress_level:Q', scale=alt.Scale(scheme='oranges'), legend=None)
).transform_filter(
    brush # Aqui está a mágica: o gráfico de barras só usa os dados filtrados pelo brush
).properties(
    width=400,
    height=350,
    title='Produtividade Média por Estresse (Dados Selecionados)'
)

# 4. Combinar os gráficos lado a lado e adicionar um título geral
painel_interativo = (scatter | bar).properties(
    title=alt.TitleParams(
        text="Análise Interativa: Desempenho, Produtividade e Estresse",
        subtitle="Selecione (arraste o mouse) no gráfico da esquerda para ver o perfil de estresse desses alunos à direita.",
        anchor='middle',
        fontSize=18,
        subtitleFontSize=14,
        dy=-10 # Ajuste de espaçamento do título
    )
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
)

# Renderizar o gráfico
painel_interativo

## O que Rouba a Atenção dos Melhores e dos Piores Alunos?

In [ ]:
# A. Criar a coluna de Total de Distração
df['total_distraction_hours'] = (
    df['phone_usage_hours'] + 
    df['social_media_hours'] + 
    df['youtube_hours'] + 
    df['gaming_hours']
)

# B. Agrupar o total de distração em "faixas" para facilitar a visualização no Boxplot
df['distraction_bins'] = pd.cut(
    df['total_distraction_hours'], 
    bins=[0, 5, 10, 15, 20, 100], 
    labels=['0-5h', '5-10h', '10-15h', '15-20h', '20h+']
)

# C. Dividir os alunos em 4 grupos (Quartis) baseados na Nota Final
df['grade_quartile'] = pd.qcut(
    df['final_grade'], 
    q=4, 
    labels=['1º Quartil (Piores Notas)', '2º Quartil', '3º Quartil', '4º Quartil (Melhores Notas)']
)


# Para o Altair empilhar as cores, precisamos "derreter" (melt) as colunas de distração
# transformando-as de várias colunas para apenas duas: 'Tipo de Distração' e 'Horas'
df_melted = df.melt(
    id_vars=['student_id', 'grade_quartile'], 
    value_vars=['phone_usage_hours', 'social_media_hours', 'youtube_hours', 'gaming_hours'],
    var_name='distraction_type', 
    value_name='hours'
)

# Renomear os tipos para o gráfico ficar mais apresentável
df_melted['distraction_type'] = df_melted['distraction_type'].replace({
    'phone_usage_hours': 'Celular',
    'social_media_hours': 'Redes Sociais',
    'youtube_hours': 'YouTube',
    'gaming_hours': 'Jogos'
})

# Gráfico 1: Boxplot da Nota Final pelo Total de Horas de Distração
boxplot = alt.Chart(df).mark_boxplot(extent='min-max', size=40).encode(
    x=alt.X('distraction_bins:O', title='Total de Distração (Horas/Dia)'),
    y=alt.Y('final_grade:Q', title='Nota Final', scale=alt.Scale(zero=False)),
    color=alt.Color('distraction_bins:O', legend=None, scale=alt.Scale(scheme='teals'))
).properties(
    width=350,
    height=350,
    title='Impacto do Volume de Distração na Nota'
)

# Gráfico 2: Composição da Distração por Desempenho (Stacked Bar)
stacked_bar = alt.Chart(df_melted).mark_bar().encode(
    x=alt.X('grade_quartile:O', title='Desempenho dos Alunos (Quartis)'),
    y=alt.Y('mean(hours):Q', title='Média de Horas Diárias'),
    color=alt.Color('distraction_type:N', title='Tipo de Distração', scale=alt.Scale(scheme='tableau10')),
    tooltip=[
        alt.Tooltip('grade_quartile', title='Grupo'), 
        alt.Tooltip('distraction_type', title='Distração'), 
        alt.Tooltip('mean(hours)', title='Média de Horas', format='.1f')
    ]
).properties(
    width=350,
    height=350,
    title='Do que os alunos se distraem?'
)

# Combinar os gráficos lado a lado
painel_distracao = (boxplot | stacked_bar).properties(
    title=alt.TitleParams(
        text="Análise de Distrações: Volume vs. Composição",
        subtitle="Verificando se o acúmulo de distrações derruba as notas e os tipos favoritos de distração por nível de aluno.",
        anchor='middle',
        fontSize=18,
        subtitleFontSize=14,
        dy=-15
    )
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
)

# Exibir o painel
painel_distracao